# 🎬 Movie Recommendation System using ANN (Autoencoder)

**Dataset:** MovieLens `movies.csv` → Download from [grouplens.org](https://grouplens.org/datasets/movielens/latest/) → use `ml-latest-small` zip

| Salary Prediction | Movie Recommendation |
|---|---|
| Regression → predict a number | Similarity → find closest movies |
| Output: 1 neuron (salary) | Output: Autoencoder (compressed features) |
| Loss: MSE on salary | Loss: MSE on reconstruction |
| Supervised | Unsupervised (Autoencoder) |

In [1]:
# ───────────────────────────────────────────────
# CELL 1 — Imports
# CHANGED: added cosine_similarity, removed LabelEncoder
# ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ───────────────────────────────────────────────
# CELL 2 — Load data
# CHANGED: movies.csv from MovieLens (columns: movieId, title, genres)
# ───────────────────────────────────────────────
df = pd.read_csv('movies.csv')   # columns: movieId, title, genres
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
# ───────────────────────────────────────────────
# CELL 3 — Check nulls  (same as before)
# ───────────────────────────────────────────────
df.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [4]:
# ───────────────────────────────────────────────
# CELL 4 — Drop nulls + remove movies with no genre
# ───────────────────────────────────────────────
df.dropna(inplace=True)
df = df[df['genres'] != '(no genres listed)']
df.reset_index(drop=True, inplace=True)
df.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [5]:
# ───────────────────────────────────────────────
# CELL 5 — One-Hot Encode genres
# CHANGED: No LabelEncoder needed
# genres column looks like: "Action|Comedy|Drama"
# str.get_dummies splits on '|' and creates one column per genre
# ───────────────────────────────────────────────
genres_dummies = df['genres'].str.get_dummies(sep='|')  # one-hot per genre
print('All genres found:')
print(genres_dummies.columns.tolist())
print('\nShape:', genres_dummies.shape)

All genres found:
['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

Shape: (9708, 19)


In [6]:
# ───────────────────────────────────────────────
# CELL 6 — Feature matrix
# CHANGED: No train/test split — Autoencoder is UNSUPERVISED
# Input = Output = genre features (model learns to reconstruct itself)
# ───────────────────────────────────────────────
X = genres_dummies.values.astype(np.float32)
print('Feature matrix shape:', X.shape)   # (num_movies, num_genres)

Feature matrix shape: (9708, 19)


In [7]:
# ───────────────────────────────────────────────
# CELL 7 — Scale features
# Fit on ALL data — no leakage risk since this is unsupervised
# ───────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaled shape:', X_scaled.shape)

Scaled shape: (9708, 19)


In [8]:
# ───────────────────────────────────────────────
# CELL 8 — Convert to tensor
# CHANGED: Only ONE tensor (no y tensor — input IS the target)
# ───────────────────────────────────────────────
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
print('Tensor shape:', X_tensor.shape)

Tensor shape: torch.Size([9708, 19])


In [9]:
# ───────────────────────────────────────────────
# CELL 9 — Custom Dataset
# CHANGED: __getitem__ returns only features (no label)
# Because in Autoencoder: input == target
# ───────────────────────────────────────────────
class MovieDataset(Dataset):

    def __init__(self, features):
        self.features = features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index]   # ✅ no label returned

In [10]:
# ───────────────────────────────────────────────
# CELL 10 — Create dataset object
# ───────────────────────────────────────────────
movie_dataset = MovieDataset(X_tensor)
print('Sample item shape:', movie_dataset[0].shape)
movie_dataset[0]

Sample item shape: torch.Size([19])


tensor([-0.4816,  2.5858,  3.8586,  3.6906,  1.2588, -0.3754, -0.2179, -0.9031,
         3.3856, -0.0951, -0.3347, -0.1286, -0.1888, -0.2505, -0.4436, -0.3351,
        -0.4923, -0.2024, -0.1323])

In [11]:
# ───────────────────────────────────────────────
# CELL 11 — DataLoader
# ───────────────────────────────────────────────
movie_loader = DataLoader(movie_dataset, batch_size=64, shuffle=True)

In [12]:
# ───────────────────────────────────────────────
# CELL 12 — Autoencoder Model  (COMPLETELY CHANGED)
#
# Old salary model:  features → [128 → 64 → 32 → 1]
#
# New Autoencoder:
#   Encoder: features → 128 → 64 → latent_dim  (compress)
#   Decoder: latent_dim → 64 → 128 → features  (reconstruct)
#
# After training, only the ENCODER is used to get movie fingerprints
# ───────────────────────────────────────────────
class MovieAutoencoder(nn.Module):

    def __init__(self, num_features, latent_dim=32):
        super().__init__()

        # Encoder: compress genres → small fingerprint
        self.encoder = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)   # ✅ bottleneck (32-dim fingerprint)
        )

        # Decoder: fingerprint → reconstruct genres
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, num_features)  # ✅ same size as input
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded          # ✅ compared vs original x during training

    def encode(self, x):
        return self.encoder(x)  # ✅ used AFTER training to get embeddings

In [13]:
# ───────────────────────────────────────────────
# CELL 13 — Hyperparameters
# ───────────────────────────────────────────────
epochs        = 100
learning_rate = 0.001
latent_dim    = 32      # size of compressed movie fingerprint
num_features  = X_tensor.shape[1]

print(f'Input features  : {num_features}')
print(f'Latent dim      : {latent_dim}')
print(f'Epochs          : {epochs}')
print(f'Learning rate   : {learning_rate}')

Input features  : 19
Latent dim      : 32
Epochs          : 100
Learning rate   : 0.001


In [14]:
# ───────────────────────────────────────────────
# CELL 14 — Model, Loss, Optimizer
# ───────────────────────────────────────────────
model     = MovieAutoencoder(num_features=num_features, latent_dim=latent_dim)

loss_fn   = nn.MSELoss()     # ✅ reconstruction loss (same as salary)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

print(model)

MovieAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=19, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=19, bias=True)
  )
)


In [15]:
# ───────────────────────────────────────────────
# CELL 15 — Training loop
# CHANGED: target = input batch itself (not salary labels)
#          loss = how well decoder reconstructs the input
# ───────────────────────────────────────────────
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch in movie_loader:         # batch = features only (no labels)

        reconstructed = model(batch)               # forward pass

        loss = loss_fn(reconstructed, batch)       # ✅ output vs INPUT

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(movie_loader)

    if (epoch + 1) % 10 == 0:
        print(f'Epoch: {epoch + 1:3d}/{epochs},  Reconstruction Loss: {avg_loss:.6f}')

Epoch:  10/100,  Reconstruction Loss: 0.004157
Epoch:  20/100,  Reconstruction Loss: 0.002107
Epoch:  30/100,  Reconstruction Loss: 0.001942
Epoch:  40/100,  Reconstruction Loss: 0.001427
Epoch:  50/100,  Reconstruction Loss: 0.001638
Epoch:  60/100,  Reconstruction Loss: 0.001064
Epoch:  70/100,  Reconstruction Loss: 0.001127
Epoch:  80/100,  Reconstruction Loss: 0.000944
Epoch:  90/100,  Reconstruction Loss: 0.000824
Epoch: 100/100,  Reconstruction Loss: 0.000781


In [16]:
# ───────────────────────────────────────────────
# CELL 16 — Get ALL movie embeddings
# CHANGED: No MAE evaluation
# Run encoder on all movies → gives each movie a 32-dim fingerprint
# These fingerprints are used for similarity search
# ───────────────────────────────────────────────
model.eval()

with torch.inference_mode():
    all_embeddings = model.encode(X_tensor).numpy()   # shape: (num_movies, 32)

print('All embeddings shape:', all_embeddings.shape)

All embeddings shape: (9708, 32)


In [17]:
# ───────────────────────────────────────────────
# CELL 17 — List all movie titles with index
# CHANGED: replaces encoder.classes_ check from salary code
# Use this to find exact movie name for recommendation
# ───────────────────────────────────────────────
for i, title in enumerate(df['title'].values):
    print(f'{i:5d} → {title}')

    0 → Toy Story (1995)
    1 → Jumanji (1995)
    2 → Grumpier Old Men (1995)
    3 → Waiting to Exhale (1995)
    4 → Father of the Bride Part II (1995)
    5 → Heat (1995)
    6 → Sabrina (1995)
    7 → Tom and Huck (1995)
    8 → Sudden Death (1995)
    9 → GoldenEye (1995)
   10 → American President, The (1995)
   11 → Dracula: Dead and Loving It (1995)
   12 → Balto (1995)
   13 → Nixon (1995)
   14 → Cutthroat Island (1995)
   15 → Casino (1995)
   16 → Sense and Sensibility (1995)
   17 → Four Rooms (1995)
   18 → Ace Ventura: When Nature Calls (1995)
   19 → Money Train (1995)
   20 → Get Shorty (1995)
   21 → Copycat (1995)
   22 → Assassins (1995)
   23 → Powder (1995)
   24 → Leaving Las Vegas (1995)
   25 → Othello (1995)
   26 → Now and Then (1995)
   27 → Persuasion (1995)
   28 → City of Lost Children, The (Cité des enfants perdus, La) (1995)
   29 → Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
   30 → Dangerous Minds (1995)
   31 → Twelve Monkeys (a.k.a. 12 Mo

In [27]:
# ───────────────────────────────────────────────
# CELL 18 — Recommend 7 movies  (COMPLETELY CHANGED)
#
# How it works:
#   1. Find the input movie's index in df
#   2. Get its 32-dim embedding from all_embeddings
#   3. Compute cosine similarity with EVERY other movie embedding
#   4. Sort by similarity → pick top 7 (skip the movie itself)
# ───────────────────────────────────────────────
def recommend_movies(movie_title, top_n=7):

    # Step 1: find the movie index (case-insensitive partial match)
    matches = df[df['title'].str.contains(movie_title, case=False, na=False)]

    if matches.empty:
        print(f'Movie "{movie_title}" not found in dataset.')
        return

    movie_idx = matches.index[0]
    print(f'\n🎬 Query Movie : {df.loc[movie_idx, "title"]}')
    print(f'   Genres     : {df.loc[movie_idx, "genres"]}')
    print('─' * 55)

    # Step 2: get embedding of this movie
    query_embedding = all_embeddings[movie_idx].reshape(1, -1)  # (1, 32)

    # Step 3: cosine similarity vs ALL movies
    similarities = cosine_similarity(query_embedding, all_embeddings)[0]  # (num_movies,)

    # Step 4: sort descending, skip the query movie itself
    similar_indices = np.argsort(similarities)[::-1]
    similar_indices = [i for i in similar_indices if i != movie_idx][:top_n]

    # Step 5: display recommendations
    print(f'\n🍿 Top {top_n} Recommendations:\n')
    for rank, idx in enumerate(similar_indices, 1):
        print(f'  {rank}. {df.loc[idx, "title"]}')
        print(f'     Genres    : {df.loc[idx, "genres"]}')
        print(f'     Similarity: {similarities[idx]:.4f}')
        print()


# ─── TYPE YOUR MOVIE HERE ───
recommend_movies("Superman")


🎬 Query Movie : Superman (1978)
   Genres     : Action|Adventure|Sci-Fi
───────────────────────────────────────────────────────

🍿 Top 7 Recommendations:

  1. Hulk (2003)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  2. Terminator 3: Rise of the Machines (2003)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  3. Star Wars: Episode V - The Empire Strikes Back (1980)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  4. Independence Day: Resurgence (2016)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  5. Star Trek Beyond (2016)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  6. Rocketeer, The (1991)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000

  7. Power/Rangers (2015)
     Genres    : Action|Adventure|Sci-Fi
     Similarity: 1.0000



In [23]:
# ───────────────────────────────────────────────
# CELL 19 — Try more movies!
# ───────────────────────────────────────────────
recommend_movies("Batman")


🎬 Query Movie : Batman Forever (1995)
   Genres     : Action|Adventure|Comedy|Crime
───────────────────────────────────────────────────────

🍿 Top 7 Recommendations:

  1. Rumble in the Bronx (Hont faan kui) (1995)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  2. Kingsman: The Secret Service (2015)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  3. Crime Busters (1977)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  4. Nothing to Lose (1997)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  5. It's a Mad, Mad, Mad, Mad World (1963)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  6. I Spy (2002)
     Genres    : Action|Adventure|Comedy|Crime
     Similarity: 1.0000

  7. Flashback (1990)
     Genres    : Action|Adventure|Comedy|Crime|Drama
     Similarity: 0.9467



## 📊 Key Changes from Salary Code

```
Salary Code          →   Movie Recommendation
─────────────────────────────────────────────
LabelEncoder         →   get_dummies (genre one-hot)
Train/Test Split     →   No split (unsupervised)
Regression model     →   Autoencoder (encoder + decoder)
y = Salary           →   y = X itself (reconstruction loss)
MAE evaluation       →   Cosine similarity search
Predict 1 number     →   Find top-7 similar embeddings
```

The **bottleneck layer** (`latent_dim=32`) forces the ANN to learn *what makes movies similar* in compressed space — exactly like PCA compression! 🎯